In [ ]:
import open3d as o3d
import numpy as np

def read_ply_file(file_path):
    point_cloud = o3d.io.read_point_cloud(file_path)
    return point_cloud

file_path = 'centered.ply'
point_cloud = read_ply_file(file_path)

vertices = np.asarray(point_cloud.points)

distances = np.linalg.norm(vertices, axis=1)

normalized_distances = (distances - np.min(distances)) / (np.max(distances) - np.min(distances))

colors = np.repeat(normalized_distances[:, np.newaxis], 3, axis=1)
colors = 1 - colors

point_cloud.colors = o3d.utility.Vector3dVector(colors)

point_cloud = point_cloud.voxel_down_sample(voxel_size=0.05)

point_cloud.estimate_normals()
point_cloud.orient_normals_consistent_tangent_plane(k=20)

distances = point_cloud.compute_nearest_neighbor_distance()
avg_dist = np.mean(distances)
radius = 3 * avg_dist

radii = [radius, radius * 1.5]
mesh = o3d.geometry.TriangleMesh.create_from_point_cloud_ball_pivoting(
    point_cloud,
    o3d.utility.DoubleVector(radii))

mesh = mesh.filter_smooth_taubin(number_of_iterations=10)
mesh.compute_triangle_normals()
mesh.compute_vertex_normals()

o3d.visualization.draw_geometries(
    [mesh],
    mesh_show_back_face=True
)


Ez itt fent 7 percces kb és egészen pontos surfacet csinál.

GPU-s:

jó ez, csak 20 perc

In [4]:
import open3d as o3d
import numpy as np
import cupy as cp
from numba import njit, prange

@njit(parallel=True)
def compute_colors_numba(vertices):
    min_dist = np.min(vertices)
    max_dist = np.max(vertices)
    colors = np.empty((vertices.shape[0], 3))
    for i in prange(vertices.shape[0]):
        val = 1 - (vertices[i] - min_dist) / (max_dist - min_dist)
        colors[i] = (val, val, val)
    return colors

def compute_colors_gpu(vertices):
    vertices_gpu = cp.asarray(vertices)
    min_dist = cp.min(vertices_gpu)
    max_dist = cp.max(vertices_gpu)
    normalized = 1 - (vertices_gpu - min_dist) / (max_dist - min_dist)
    colors = cp.stack((normalized, normalized, normalized), axis=1)
    return cp.asnumpy(colors)

input_file = "centered.ply"
pcd = o3d.io.read_point_cloud(input_file)
vertices = np.asarray(pcd.points)

distances = cp.linalg.norm(cp.asarray(vertices), axis=1).get()
colors = compute_colors_gpu(distances)
pcd.colors = o3d.utility.Vector3dVector(colors)

pcd.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.1, max_nn=30))
pcd.orient_normals_consistent_tangent_plane(k=30)

distances = pcd.compute_nearest_neighbor_distance()
avg_dist = np.mean(distances)
radii = o3d.utility.DoubleVector([avg_dist*3, avg_dist*6])

mesh = o3d.geometry.TriangleMesh.create_from_point_cloud_ball_pivoting(pcd, radii)
o3d.visualization.draw_geometries([mesh], mesh_show_back_face=True)